# EfficientNetV2-S — Trening z oceną na nowych perspektywach

## Strategia podziału danych

| Zbiór | Źródło | Rola |
|-------|--------|------|
| **train** | `train1` (kamery `orb`) | Trening modelu |
| **val** | obserwacje z `train2` których `image_id` **nie** występuje w `train1` (~4.5%) | Walidacja na nowych perspektywach (`tur`) zbliżonych do testu |

Model uczy się na perspektywie fisheye (`orb`), a oceniamy go na kamerach `tur` — tych samych co zbiór testowy konkursu.

In [ ]:
import torch
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Wymuś inicjalizację CUDA od razu
_ = torch.zeros(1).cuda()
torch.cuda.synchronize()

# Sprawdź wynik
import time
x = torch.randn(10000, 10000, device="cuda")
t0 = time.time()
y = x @ x
torch.cuda.synchronize()
print(f"Test: {time.time()-t0:.3f}s")

In [3]:
import os, ast, copy, re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── Stałe ─────────────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: "Lateral_lying_left",
    1: "Lateral_lying_right",
    2: "Sitting",
    3: "Standing",
    4: "Sternal_lying",
}
NUM_CLASSES = len(CLASS_NAMES)

BASE_DIR = Path(r"C:/Users/Mateu/Github/Świnie/multiview_pig_posture_recognition")
TRAIN1_IMGS = BASE_DIR / "train1_images"
TRAIN2_IMGS = BASE_DIR / "train2_images"
TEST_IMGS = BASE_DIR / "test_images"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 7
SAVE_PATH = r"C:/Users/Mateu/Github/Świnie/best_efficientnetv2_s_pigs_v2.pt"

print(f"Device: {DEVICE}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. DANE
# ══════════════════════════════════════════════════════════════════════════════
train1 = pd.read_csv(BASE_DIR / "train1.csv")
train2 = pd.read_csv(BASE_DIR / "train2.csv")

for df, src in [(train1, "train1"), (train2, "train2")]:
    df["source"]      = src
    df["bbox_parsed"] = df["bbox"].apply(ast.literal_eval)
    df["class_name"]  = df["class_id"].map(CLASS_NAMES)

# VAL = obserwacje z train2, których image_id nie ma w train1
train1_ids = set(train1["image_id"].unique())
val_mask   = ~train2["image_id"].isin(train1_ids)

train_df = train1.reset_index(drop=True)
val_df   = train2[val_mask].reset_index(drop=True)

assert len(set(train_df["image_id"]) & set(val_df["image_id"])) == 0, "Przecięcie image_id!"

print(f"Train: {len(train_df):,} obs  |  {train_df['image_id'].nunique():,} zdjęć")
print(f"Val:   {len(val_df):,} obs  |  {val_df['image_id'].nunique():,} zdjęć  "
      f"({len(val_df)/len(train2)*100:.1f}% train2)")
print("\nRozkład klas — train:"); print(train_df["class_name"].value_counts().to_string())
print("\nRozkład klas — val:");   print(val_df["class_name"].value_counts().to_string())

# ══════════════════════════════════════════════════════════════════════════════
# 2. FUNKCJE POMOCNICZE + DATASET
# ══════════════════════════════════════════════════════════════════════════════
def load_image(image_id, source):
    folder = {"train1": TRAIN1_IMGS, "train2": TRAIN2_IMGS, "test": TEST_IMGS}[source]
    return Image.open(folder / image_id).convert("RGB")


def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    """bbox = [x_topleft, y_topleft, w, h]"""
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w * padding;      y1 = y - h * padding
    x2 = x + w + w * padding;  y2 = y + h + h * padding
    if make_square:
        side   = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2, y1, y2 = cx - side/2, cx + side/2, cy - side/2, cy + side/2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((max(x1, 0), max(y1, 0), max(x2, x1+1), max(y2, y1+1)))


class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.05, p=0.3):
        self.mean, self.std, self.p = mean, std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(tensor + torch.randn_like(tensor) * self.std + self.mean, 0., 1.)
        return tensor


class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df, self.transform = df.reset_index(drop=True), transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = load_image(row["image_id"], row["source"])
        crop  = crop_with_padding(image, row["bbox_parsed"], padding=0.12, make_square=True)
        if self.transform:
            crop = self.transform(crop)
        return crop, int(row["class_id"])

# ══════════════════════════════════════════════════════════════════════════════
# 3. TRANSFORMY — dokładnie jak w EfficientNet_V2.ipynb
# ══════════════════════════════════════════════════════════════════════════════
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),
        scale=(0.9, 1.1)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.03
    ),
    transforms.ToTensor(),
    AddGaussianNoise(p=0.2, std=0.03),
    transforms.RandomErasing(
        p=0.2,
        scale=(0.01, 0.05),
        ratio=(0.75, 1.33),
        value="random"
    ),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ══════════════════════════════════════════════════════════════════════════════
# 4. DATALOADERS + WEIGHTED SAMPLER
# ══════════════════════════════════════════════════════════════════════════════
train_targets  = train_df["class_id"].values
class_counts   = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights  = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_targets])
print("\nLiczebność klas (train):", class_counts)

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    PigPostureDataset(train_df, transform=train_transform),
    batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False
)
val_loader = DataLoader(
    PigPostureDataset(val_df, transform=val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. MODEL
# ══════════════════════════════════════════════════════════════════════════════
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)
print(f"\nModel gotowy | parametry: {sum(p.numel() for p in model.parameters()):,}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. PĘTLA TRENINGOWA
# ══════════════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, epoch):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    n_batches = len(loader)

    for batch_i, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Pasek postępu
        filled = int(30 * (batch_i + 1) / n_batches)
        bar    = "█" * filled + "░" * (30 - filled)
        pct    = 100 * (batch_i + 1) / n_batches
        print(f"\r  [train] |{bar}| {pct:5.1f}%  {batch_i+1}/{n_batches}  loss={loss.item():.4f}",
              end="", flush=True)

    print()
    n = len(loader.dataset)
    return total_loss / n, accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average="macro")


def evaluate(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return total_loss / n, accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average="macro"), all_labels, all_preds



best_f1, best_state, history = 0.0, None, []

for epoch in range(EPOCHS):

    t0 = time.time()
    tr_loss, tr_acc, tr_f1                    = train_one_epoch(model, loader=train_loader, epoch=epoch)
    val_loss, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)
    scheduler.step(val_f1)
    elapsed = time.time() - t0

    history.append({"epoch": epoch+1,
                    "train_loss": tr_loss, "train_acc": tr_acc, "train_f1": tr_f1,
                    "val_loss":  val_loss, "val_acc":  val_acc, "val_f1":  val_f1})

    flag = "  ← best" if val_f1 > best_f1 else ""
    print(f"Epoch {epoch+1:>2}/{EPOCHS} ({elapsed/60:.1f} min) | "
          f"train loss {tr_loss:.4f}  acc {tr_acc:.4f}  f1 {tr_f1:.4f} | "
          f"val loss {val_loss:.4f}  acc {val_acc:.4f}  f1 {val_f1:.4f}"
          f"  lr {optimizer.param_groups[0]['lr']:.1e}{flag}", flush=True)

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = copy.deepcopy(model.state_dict())
        torch.save({"model_state_dict": best_state, "class_names": CLASS_NAMES,
                    "best_val_f1": best_f1, "epoch": epoch+1}, SAVE_PATH)

print(f"\nNajlepsze val macro-F1: {best_f1:.4f}")
print(f"Model zapisany → {SAVE_PATH}")

Device: cuda
Train: 22,934 obs  |  3,090 zdjęć
Val:   516 obs  |  60 zdjęć  (2.2% train2)

Rozkład klas — train:
class_name
Standing               9617
Sternal_lying          6208
Lateral_lying_right    3376
Lateral_lying_left     3053
Sitting                 680

Rozkład klas — val:
class_name
Standing               311
Sternal_lying          101
Lateral_lying_right     59
Lateral_lying_left      30
Sitting                 15

Liczebność klas (train): [3053 3376  680 9617 6208]
Train batches: 717  |  Val batches: 17

Model gotowy | parametry: 20,183,893
  [GPU check epoch 0] mnożenie 10k×10k: 8.265s  ✗ bardzo wolno!
  [train] |░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░|   1.1%  8/717  loss=1.5255

KeyboardInterrupt: 

In [ ]:
# ── Załaduj najlepszy stan ─────────────────────────────────────────────────────
model.load_state_dict(best_state)
_, val_acc, val_f1, y_true, y_pred = evaluate(model, val_loader)

target_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
print(f"Najlepszy model — val (nowe perspektywy)")
print(f"Accuracy: {val_acc:.4f}  |  Macro F1: {val_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=target_names))

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Loss ──────────────────────────────────────────────────────────────────────
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", markersize=4, label="train")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"],   marker="o", markersize=4, label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

# ── Macro F1 ──────────────────────────────────────────────────────────────────
axes[1].plot(hist_df["epoch"], hist_df["train_f1"], marker="o", markersize=4, label="train")
axes[1].plot(hist_df["epoch"], hist_df["val_f1"],   marker="o", markersize=4, label="val")
axes[1].axhline(best_f1, color="red", linestyle="--", linewidth=1,
                label=f"best val F1 = {best_f1:.4f}")
axes[1].set_title("Macro F1"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
im = axes[2].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[2].set_xticks(range(NUM_CLASSES)); axes[2].set_yticks(range(NUM_CLASSES))
axes[2].set_xticklabels(target_names, rotation=40, ha="right", fontsize=8)
axes[2].set_yticklabels(target_names, fontsize=8)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
axes[2].set_title(f"Confusion Matrix  —  val Macro F1 = {val_f1:.4f}")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        col = "white" if cm_norm[i, j] > 0.55 else "black"
        axes[2].text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})",
                     ha="center", va="center", fontsize=8, color=col)

plt.suptitle("EfficientNetV2-S — trening: train1 (orb)  |  walidacja: nowe perspektywy (tur)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("training_results_v2.png", dpi=150, bbox_inches="tight")
plt.show()